# Smallholder Fairness — Deep-Dive Experiments

Goal: take the Phase 3 finding (smallholder recall ≈ 0.23 vs medium 0.32 / large 0.30) and **diagnose the mechanism**.

Six controlled experiments share the same train/test split and report **per-stratum recall** so we can compare on a single axis.

| Exp | Intervention | What it isolates |
|---|---|---|
| **E0 Baseline** | Standard mean+p90 features over full 1 km AOI | Reproduces Phase 3 fairness gap |
| **E1 Pixel count** | Histogram parcel-size-in-pixels by stratum | Confirms smallholders occupy few pixels (necessary precondition) |
| **E2 Center crop** | Features from only the central 50×50 of the AOI | Tests boundary / edge contamination |
| **E3 Resolution (S2 substitute)** | Mean-pool the 10 m stack to 20 m and 30 m, then extract features | **Tests whether the gap is fundamentally about resolution.** Substitutes for PlanetScope. |
| **E4 Aggregation** | Replace mean+p90 with max + fraction-of-pixels-above-NDVI-0.5 | Tests if mean-aggregation dilutes a few irrigated pixels |
| **E5 Reweighting** | Retrain baseline RF with smallholder sample_weight = 3× and 5× | Tests training imbalance |
| **E6 Threshold** | Per-stratum optimal decision threshold from validation set | Tests if a single global threshold is miscalibrated |

All runs use the **same site-grouped train/test split** (no spatial leakage). Outputs land in `/kaggle/working/outputs/smallholder/`.

## How to use
1. Upload this notebook to a new Kaggle notebook.
2. **Add Input → Datasets**: attach `wenhaolu49/zambia-smallholder-irrigation-sentinel2-part1` AND `…-part2`.
3. Settings: Accelerator **None**, Internet **On**.
4. **Save Version → Save & Run All**. ~1–2 hours.
5. Download outputs via `kaggle kernels output <username>/<id> -p .`.

## 0 · Setup

In [ ]:
import sys, subprocess
for p in ["rasterio", "scikit-learn", "seaborn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)

import os, glob, json, math, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import precision_recall_fscore_support, average_precision_score, roc_auc_score
warnings.filterwarnings("ignore")

RNG = 42
OUT = Path("/kaggle/working/outputs/smallholder") if os.path.exists("/kaggle/working") else Path("outputs/smallholder")
OUT.mkdir(parents=True, exist_ok=True)
ON_KAGGLE = os.path.exists("/kaggle/input")
print("Output:", OUT, "  On Kaggle:", ON_KAGGLE)

## 1 · Find the Sentinel-2 stacks and the label CSV

In [ ]:
SEARCH_ROOTS = glob.glob("/kaggle/input/*") if ON_KAGGLE else ["data/features/sentinel2"]
STACKS = sorted(set(sum(
    [glob.glob(os.path.join(r, "**", "*_stack_masked.tif"), recursive=True) for r in SEARCH_ROOTS], []
)))
print(f"Found {len(STACKS)} masked stacks")
assert STACKS, "Attach the Kaggle datasets first."
print("sample:", STACKS[0])

LABEL_CANDIDATES = []
for r in SEARCH_ROOTS:
    LABEL_CANDIDATES += glob.glob(os.path.join(r, "**", "latest_irrigation_table.csv"), recursive=True)
LABEL_CANDIDATES += [
    "data/labels/labeled_surveys/random_sample/latest_irrigation_table.csv",
    "/kaggle/working/latest_irrigation_table.csv",
]
csv_path = next((p for p in LABEL_CANDIDATES if os.path.exists(p)), None)
if csv_path is None:
    import requests
    url = ("https://raw.githubusercontent.com/AIscend-Research/smallholder-irrigation-dataset/"
           "main/data/labels/labeled_surveys/random_sample/latest_irrigation_table.csv")
    csv_path = "/kaggle/working/latest_irrigation_table.csv"
    open(csv_path, "wb").write(requests.get(url, timeout=60).content)
raw_labels = pd.read_csv(csv_path)
print("label rows:", raw_labels.shape)

## 2 · Dedupe labels + define farm-size strata

Same labeler priority as Phase 3 (KL > MV > DSB > JL). Farm-size strata from `poly_avg_size_hc`.

In [ ]:
def file_id(r):
    sid = str(r["site_id"]).replace("id_", "")
    return f"{sid}_{int(r['year'])}.{int(r['month']):02d}.{int(r['day']):02d}"
raw_labels["file_id"] = raw_labels.apply(file_id, axis=1)
raw_labels["target"]  = (raw_labels["irrigation"] >= 2).astype(int)

PRIO = {"KL":5, "MV":4, "DSB":3, "JL":2, "PS":1, "AB":1}
raw_labels["_prio"] = raw_labels["operator_initials"].map(PRIO).fillna(0)
labels = (raw_labels.sort_values("_prio", ascending=False)
                    .drop_duplicates("file_id", keep="first")
                    .drop(columns="_prio").reset_index(drop=True))
print(f"deduped: {len(labels)} unique site+date images")

# Heuristic on poly_avg_size_hc units: if values are tiny (<10), they're hectares;
# if large (>>100), they're square metres. Convert to hectares.
_med = labels.loc[labels["target"]==1, "poly_avg_size_hc"].median()
size_unit = "hectares" if _med < 100 else "square_metres"
labels["poly_avg_size_ha"] = labels["poly_avg_size_hc"].astype(float)
if size_unit == "square_metres":
    labels["poly_avg_size_ha"] = labels["poly_avg_size_ha"] / 10_000.0
print(f"detected size unit: {size_unit}  (median positive size = {_med:.3g})")

def stratum(row):
    if row["target"] == 0:
        return "none"
    s = row["poly_avg_size_ha"]
    if pd.isna(s) or s <= 0: return "unknown"
    if s < 2:   return "smallholder"
    if s < 10:  return "medium"
    return "large"
labels["stratum"] = labels.apply(stratum, axis=1)
print("stratum counts (positives only):")
print(labels[labels["target"]==1]["stratum"].value_counts())

# Keep only labels with a matching downloaded stack
have = {os.path.basename(p).replace("_stack_masked.tif",""): p for p in STACKS}
labels = labels[labels["file_id"].isin(have)].reset_index(drop=True)
labels["stack_path"] = labels["file_id"].map(have)
print(f"labels matched to stacks: {len(labels)}")

## 3 · One-pass multi-variant feature extraction

For each stack (10 bands × 42 windows × 100×100), compute per-window scalar summaries under **8 different spatial-aggregation variants**, then summarise each time series temporally. Cached to disk for re-runs.

In [ ]:
S2_BANDS = ["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12"]
NWIN = 42
BAND_IDX = {b:i for i,b in enumerate(S2_BANDS)}

VARIANTS = ["full_mean","full_p90","center_mean","center_p90",
            "d20m_mean","d30m_mean","full_max","full_frac_high"]
INDICES  = ["ndvi","ndwi","evi"]
TIME_AGG = ["mean","std","max","min","amp"]

def safe_nanpercentile(a, q, axis):
    return np.where(np.isnan(a).all(axis=axis), np.nan, np.nanpercentile(np.where(np.isnan(a), -1e9, a), q, axis=axis))

def compute_indices_per_window(stack_arr):
    """stack_arr: (420, 100, 100) uint16, 0=nodata. Returns dict idx->(42,100,100) float32 NaNs for nodata."""
    s = stack_arr.reshape(10, NWIN, 100, 100).astype(np.float32)
    s = np.where(s == 0, np.nan, s)
    B,G,R,NIR = s[BAND_IDX["B2"]], s[BAND_IDX["B3"]], s[BAND_IDX["B4"]], s[BAND_IDX["B8"]]
    NDVI = (NIR - R) / (NIR + R + 1e-6)
    NDWI = (G - NIR) / (G + NIR + 1e-6)
    EVI  = 2.5 * (NIR - R) / (NIR + 6*R - 7.5*B + 1.0 + 1e-6)
    return {"ndvi":NDVI, "ndwi":NDWI, "evi":EVI}

def spatial_variants(idx_42hw):
    """Return dict variant -> (42,) per-window scalar."""
    out = {}
    flat = idx_42hw.reshape(NWIN, -1)                # full
    out["full_mean"]      = np.nanmean(flat, axis=1)
    out["full_p90"]       = safe_nanpercentile(flat, 90, axis=1)
    out["full_max"]       = np.nanmax(np.where(np.isnan(flat), -1e9, flat), axis=1)
    out["full_max"]       = np.where(out["full_max"] <= -1e8, np.nan, out["full_max"])
    out["full_frac_high"] = np.nanmean((flat > 0.5).astype(float), axis=1)
    c = idx_42hw[:, 25:75, 25:75].reshape(NWIN, -1)  # center 50x50
    out["center_mean"]    = np.nanmean(c, axis=1)
    out["center_p90"]     = safe_nanpercentile(c, 90, axis=1)
    d20 = idx_42hw.reshape(NWIN, 50, 2, 50, 2)        # 20 m pool
    out["d20m_mean"]      = np.nanmean(np.nanmean(d20, axis=4).reshape(NWIN, -1), axis=1)
    d30 = idx_42hw[:, :99, :99].reshape(NWIN, 33, 3, 33, 3)  # 30 m pool
    out["d30m_mean"]      = np.nanmean(np.nanmean(np.nanmean(d30, axis=4), axis=2).reshape(NWIN, -1), axis=1)
    return out

def temporal_summary(arr):
    return {"mean":np.nanmean(arr),"std":np.nanstd(arr),
            "max":np.nanmax(arr),"min":np.nanmin(arr),
            "amp":np.nanmax(arr)-np.nanmin(arr)}

CACHE = OUT / "features_cache.npz"
if CACHE.exists():
    z = np.load(CACHE, allow_pickle=True)
    FEATS  = z["feats"].item()    # dict[file_id] -> dict[col] -> float
    PIXELS = z["pixels"].item()   # dict[file_id] -> int (effective good pixels avg)
    print(f"loaded cached features for {len(FEATS)} images")
else:
    FEATS, PIXELS = {}, {}
    t0 = time.time()
    for k, row in enumerate(labels.itertuples(index=False)):
        with rasterio.open(row.stack_path) as src:
            arr = src.read()
        idxs = compute_indices_per_window(arr)
        feat = {}
        for iname, A in idxs.items():
            sv = spatial_variants(A)
            for vname, ts in sv.items():
                ts = np.nan_to_num(ts, nan=np.nan)
                ta = temporal_summary(ts)
                for tname, tval in ta.items():
                    feat[f"{iname}_{vname}_{tname}"] = float(tval)
        FEATS[row.file_id] = feat
        # effective good pixels: NDVI valid count averaged across windows
        valid = (~np.isnan(idxs["ndvi"])).reshape(NWIN, -1).sum(axis=1).mean()
        PIXELS[row.file_id] = float(valid)
        if (k+1) % 200 == 0:
            print(f"  processed {k+1}/{len(labels)}  ({(time.time()-t0)/60:.1f} min)")
    np.savez(CACHE, feats=FEATS, pixels=PIXELS)
    print(f"done in {(time.time()-t0)/60:.1f} min, cached to {CACHE}")

# Build feature DataFrame
feat_cols = sorted(next(iter(FEATS.values())).keys())
X_all = pd.DataFrame([FEATS[fid] for fid in labels["file_id"]], columns=feat_cols)
labels["good_pixels_avg"] = labels["file_id"].map(PIXELS)
print("feature matrix:", X_all.shape)

## 4 · Train/test split + experiment runner

Site-grouped split: every site appears in only train OR test.

In [ ]:
groups = labels["site_id"].astype(str).values
y = labels["target"].values
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RNG)
tr_idx, te_idx = next(gss.split(np.zeros(len(y)), y, groups))
print(f"train: {len(tr_idx)}  test: {len(te_idx)}")
print("test positives by stratum:")
print(labels.iloc[te_idx][labels.iloc[te_idx].target==1]["stratum"].value_counts())

def select_cols(prefix_set):
    """Return feature columns matching any of the variant prefixes."""
    return [c for c in feat_cols if any(f"_{p}_" in f"_{c}_" or c.endswith(f"_{p}_mean") or f"_{p}_" in c for p in prefix_set)]

def select_variants(variants):
    """Return feature columns for the given list of spatial variants (one or more)."""
    keep = []
    for c in feat_cols:
        for v in variants:
            if f"_{v}_" in c:
                keep.append(c); break
    return keep

def per_stratum_recall(y_true, y_pred, strata):
    out = {}
    for s in ["smallholder","medium","large"]:
        m = (strata == s) & (y_true == 1)
        if m.sum() == 0:
            out[s] = (np.nan, 0); continue
        out[s] = (y_pred[m].mean(), int(m.sum()))   # recall + n
    return out

def run_rf(feat_cols_sel, sample_weight=None, threshold=0.5, n_estimators=200, max_depth=None, label=""):
    Xtr, Xte = X_all[feat_cols_sel].fillna(0).values[tr_idx], X_all[feat_cols_sel].fillna(0).values[te_idx]
    ytr, yte = y[tr_idx], y[te_idx]
    rf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,
                                class_weight="balanced", n_jobs=-1, random_state=RNG)
    rf.fit(Xtr, ytr, sample_weight=(sample_weight[tr_idx] if sample_weight is not None else None))
    proba = rf.predict_proba(Xte)[:, 1]
    pred = (proba >= threshold).astype(int)
    strata_te = labels.iloc[te_idx]["stratum"].values
    rec_per = per_stratum_recall(yte, pred, strata_te)
    p, r, f, _ = precision_recall_fscore_support(yte, pred, average="binary", zero_division=0)
    ap = average_precision_score(yte, proba) if yte.sum() else np.nan
    res = {"label":label, "n_feats":len(feat_cols_sel),
           "precision":p, "recall":r, "f1":f, "auprc":ap}
    for s,(rec,n) in rec_per.items():
        res[f"recall_{s}"] = rec; res[f"n_{s}"] = n
    return res, proba, pred

# Baseline = mean+p90 on full AOI (Phase 3 setup)
baseline_cols = select_variants(["full_mean","full_p90"])
print(f"baseline feature count: {len(baseline_cols)}")
BASELINE, _, _ = run_rf(baseline_cols, label="E0 baseline (mean+p90, full AOI)")
print(json.dumps(BASELINE, indent=2, default=str))

## E1 · Pixel-count distribution by stratum

Necessary precondition for the mixed-pixel hypothesis: smallholder parcels should occupy noticeably fewer pixels than larger ones.

In [ ]:
# Each 10 m pixel = 100 m² = 0.01 ha. So pixels = size_ha * 100.
pos = labels[labels["target"]==1].copy()
pos["pixels_est"] = pos["poly_avg_size_ha"] * 100.0
fig, ax = plt.subplots(figsize=(7,4))
for s, c in zip(["smallholder","medium","large"], ["#d62728","#ff7f0e","#2ca02c"]):
    sub = pos[pos["stratum"]==s]["pixels_est"].dropna()
    if len(sub)==0: continue
    ax.hist(np.log10(sub+1), bins=20, alpha=0.55, label=f"{s} (n={len(sub)}, median={int(sub.median())} px)", color=c)
ax.set_xlabel("log10 (estimated pixels per parcel)"); ax.set_ylabel("count")
ax.set_title("E1 — Parcel pixel count by stratum\n(necessary precondition for mixed-pixel hypothesis)")
ax.legend(); plt.tight_layout(); plt.savefig(OUT/"E1_pixel_count.png", dpi=140); plt.show()
pos.groupby("stratum")["pixels_est"].describe().to_csv(OUT/"E1_pixel_count.csv")

## E2 · Boundary effect — center-crop features

In [ ]:
center_cols = select_variants(["center_mean","center_p90"])
E2, _, _ = run_rf(center_cols, label="E2 center-crop (50×50)")
print(json.dumps(E2, indent=2, default=str))

## E3 · Resolution test — downsample S2 to 20 m and 30 m

Substitute for PlanetScope. If smallholder recall drops monotonically with coarsening, the gap is fundamentally about pixel size.

In [ ]:
E3_20, _, _ = run_rf(select_variants(["d20m_mean"]), label="E3 downsampled 20 m")
E3_30, _, _ = run_rf(select_variants(["d30m_mean"]), label="E3 downsampled 30 m")
for r in (E3_20, E3_30): print(r["label"], "smallholder recall:", round(r["recall_smallholder"],3),
                              " medium:", round(r["recall_medium"],3), " large:", round(r["recall_large"],3))

## E4 · Aggregation — max + frac-above-0.5 instead of mean+p90

In [ ]:
E4_max,  _, _ = run_rf(select_variants(["full_max","full_mean"]),         label="E4 max + mean (full AOI)")
E4_frac, _, _ = run_rf(select_variants(["full_frac_high","full_mean"]),    label="E4 frac-above-0.5 + mean")
for r in (E4_max, E4_frac): print(r["label"], "smallholder recall:", round(r["recall_smallholder"],3))

## E5 · Reweighting — boost smallholder loss weight

In [ ]:
sw = np.ones(len(y), dtype=float)
sw[(labels["stratum"]=="smallholder").values] = 3.0
E5_3x, _, _ = run_rf(baseline_cols, sample_weight=sw, label="E5 smallholder weight = 3×")
sw[(labels["stratum"]=="smallholder").values] = 5.0
E5_5x, _, _ = run_rf(baseline_cols, sample_weight=sw, label="E5 smallholder weight = 5×")
for r in (E5_3x, E5_5x): print(r["label"], "smallholder recall:", round(r["recall_smallholder"],3))

## E6 · Per-stratum threshold sweep

Use 5-fold CV on the training set to pick per-stratum thresholds, then evaluate on the held-out test set.

In [ ]:
from sklearn.model_selection import GroupKFold
Xtr_b = X_all[baseline_cols].fillna(0).values[tr_idx]
ytr_b = y[tr_idx]
groups_tr = labels.iloc[tr_idx]["site_id"].astype(str).values
strata_tr = labels.iloc[tr_idx]["stratum"].values

# Get out-of-fold validation probas
oof_proba = np.zeros(len(ytr_b))
gkf = GroupKFold(n_splits=5)
for fi, (a, b) in enumerate(gkf.split(Xtr_b, ytr_b, groups_tr)):
    rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=RNG)
    rf.fit(Xtr_b[a], ytr_b[a])
    oof_proba[b] = rf.predict_proba(Xtr_b[b])[:, 1]

thresholds = np.linspace(0.05, 0.95, 19)
best_thr = {}
for s in ["smallholder","medium","large"]:
    mask = (strata_tr == s) & (ytr_b == 1) | (ytr_b == 0)   # include all negs + this stratum's pos
    if (strata_tr == s).sum() == 0: continue
    best_f1, best_t = -1, 0.5
    for t in thresholds:
        pred = (oof_proba[mask] >= t).astype(int)
        _, _, f, _ = precision_recall_fscore_support(ytr_b[mask], pred, average="binary", zero_division=0)
        if f > best_f1: best_f1, best_t = f, t
    best_thr[s] = best_t
print("chosen per-stratum thresholds:", best_thr)

# Refit on full train, predict on test, apply per-stratum threshold
rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", n_jobs=-1, random_state=RNG)
rf.fit(Xtr_b, ytr_b)
Xte_b = X_all[baseline_cols].fillna(0).values[te_idx]
te_proba = rf.predict_proba(Xte_b)[:, 1]
strata_te = labels.iloc[te_idx]["stratum"].values
pred_strat = np.zeros_like(te_proba, dtype=int)
for i, s in enumerate(strata_te):
    t = best_thr.get(s, 0.5)
    pred_strat[i] = int(te_proba[i] >= t)
rec_per = per_stratum_recall(y[te_idx], pred_strat, strata_te)
E6 = {"label":"E6 per-stratum threshold", "n_feats":len(baseline_cols),
      "precision": precision_recall_fscore_support(y[te_idx], pred_strat, average="binary", zero_division=0)[0],
      "recall":    precision_recall_fscore_support(y[te_idx], pred_strat, average="binary", zero_division=0)[1],
      "f1":        precision_recall_fscore_support(y[te_idx], pred_strat, average="binary", zero_division=0)[2],
      "auprc": average_precision_score(y[te_idx], te_proba)}
for s,(rec,n) in rec_per.items():
    E6[f"recall_{s}"] = rec; E6[f"n_{s}"] = n
print(json.dumps(E6, indent=2, default=str))

## Summary — which intervention closes the smallholder gap?

In [ ]:
rows = [BASELINE, E2, E3_20, E3_30, E4_max, E4_frac, E5_3x, E5_5x, E6]
summary = pd.DataFrame(rows)[[
    "label","n_feats","precision","recall","f1","auprc",
    "recall_smallholder","recall_medium","recall_large",
    "n_smallholder","n_medium","n_large",
]]
summary.to_csv(OUT/"summary_per_stratum.csv", index=False)
summary

In [ ]:
# Plot: smallholder recall lift over baseline
baseline_sm = BASELINE["recall_smallholder"]
labels_short = [r["label"].split(" ", 1)[0] + " " + r["label"].split(" ", 1)[1][:32] for r in rows]
vals = [r["recall_smallholder"] for r in rows]
diffs = [v - baseline_sm for v in vals]

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#888" if d == 0 else ("#2ca02c" if d > 0 else "#d62728") for d in diffs]
ax.barh(labels_short, diffs, color=colors)
ax.axvline(0, color="k", lw=0.7)
ax.set_xlabel("Δ smallholder recall vs E0 baseline")
ax.set_title("Which intervention reduces the smallholder gap?")
for i, (d, v) in enumerate(zip(diffs, vals)):
    ax.text(d + (0.005 if d >= 0 else -0.005), i, f"{v:.2f}",
            va="center", ha="left" if d >= 0 else "right", fontsize=9)
plt.tight_layout(); plt.savefig(OUT/"summary_lift.png", dpi=140); plt.show()
print("Saved:", OUT/"summary_per_stratum.csv", OUT/"summary_lift.png")

## How to read this

- The bar chart shows **lift in smallholder recall** for each intervention vs the E0 baseline.
- Whichever intervention gives the **largest positive lift** identifies the mechanism:
  - **E2 best** → boundary / edge contamination → propose interior-only aggregation
  - **E3 monotonic degradation 10 m → 30 m** → resolution is the cause → propose resolution-aware ensemble
  - **E4 (max or frac-high) best** → mean-aggregation dilution → propose max/percentile aggregation
  - **E5 best** → class imbalance → propose stratum-weighted training
  - **E6 best** → threshold miscalibration → propose per-stratum thresholds (the lightest "fix")
- Often more than one intervention helps — combine them into a single proposed method for the paper.